In [6]:
# Install dependencies
!pip install torch torchvision matplotlib gradio

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import gradio as gr
import numpy as np

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
lr = 0.0002
batch_size = 128
image_size = 28 * 28
z_dim = 100
num_epochs = 5   # keep small for demo

# Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.MNIST(root="./data", transform=transform, download=True)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Generator
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, image_size),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(image_size, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Initialize models
gen = Generator().to(device)
disc = Discriminator().to(device)

criterion = nn.BCELoss()
opt_gen = optim.Adam(gen.parameters(), lr=lr)
opt_disc = optim.Adam(disc.parameters(), lr=lr)

# Training GAN
print("Training GAN... ⏳")

for epoch in range(num_epochs):
    for real, _ in loader:
        real = real.view(-1, image_size).to(device)
        batch_size_curr = real.shape[0]

        # Train Discriminator
        noise = torch.randn(batch_size_curr, z_dim).to(device)
        fake = gen(noise)

        disc_real = disc(real).view(-1)
        loss_real = criterion(disc_real, torch.ones_like(disc_real))

        disc_fake = disc(fake.detach()).view(-1)
        loss_fake = criterion(disc_fake, torch.zeros_like(disc_fake))

        loss_disc = (loss_real + loss_fake) / 2

        disc.zero_grad()
        loss_disc.backward()
        opt_disc.step()

        # Train Generator
        output = disc(fake).view(-1)
        loss_gen = criterion(output, torch.ones_like(output))

        gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed")

print("Training Done ✅")

# Function to generate images
def generate_images(num_images):
    num_images = int(num_images)
    noise = torch.randn(num_images, z_dim).to(device)

    with torch.no_grad():
        fake = gen(noise).reshape(-1, 28, 28).cpu().numpy()

    images = []
    for img in fake:
        img = (img + 1) / 2  # Denormalize
        images.append(img)

    return images

# Gradio UI
interface = gr.Interface(
    fn=generate_images,
    inputs=gr.Slider(1, 10, step=1, label="Number of Images"),
    outputs=gr.Gallery(label="Generated Images"),
    title="🧠 GAN Image Generator (MNIST)",
    description="Generate handwritten digit images using a trained GAN model."
)

interface.launch()

100%|██████████| 9.91M/9.91M [00:00<00:00, 29.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 35.1MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 87.9MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.14MB/s]


Training GAN... ⏳
Epoch 1/5 completed
Epoch 2/5 completed
Epoch 3/5 completed
Epoch 4/5 completed
Epoch 5/5 completed
Training Done ✅
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f532c9221a70092517.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
